# AKORI · Notebook BUILD (à lancer une seule fois)

Ce notebook fait **tout ce qu'il ne faut pas refaire pendant la démonstration** :

1. télécharge le modèle d'embeddings (BGE-M3) et vérifie qu'il fonctionne ;
2. indexe chaque PDF du dossier `MyDrive/AKORI/a_indexer/` (extraction, découpage par page, vecteurs, index FAISS) ;
3. recalcule les anciens index si le modèle d'embeddings a changé ;
4. pré-génère le **résumé, les flashcards et le quiz** de chaque cours ;
5. enregistre tout sur Google Drive, dans le compte de démonstration.

Pendant la démonstration (notebook **Demo**), il ne reste que la recherche et la réponse du chat.

> Conseil : activez un GPU (Exécution → Modifier le type d'exécution → T4). L'indexation est beaucoup plus rapide.

## 1. Installation

In [ ]:
!pip -q install gradio google-genai pymupdf langchain-community langchain-text-splitters faiss-cpu sentence-transformers

## 2. Google Drive et clé Gemini

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")                         # AKORI enregistre tout dans MyDrive/AKORI

def secret(name, default=""):
    try:
        return userdata.get(name) or default
    except Exception:
        return default

os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY")        # Colab > icône clé > Secrets
assert os.environ["GEMINI_API_KEY"], "Ajoutez le secret GEMINI_API_KEY (icône clé à gauche) puis relancez la cellule."

## 3. Récupérer le code d'AKORI

In [ ]:
import os, subprocess, shutil
os.chdir("/content")                                   # sortir du dossier avant de le supprimer
REPO   = "github.com/rafalimanananomeniavoedmjohana-bit/AKORI.git"
BRANCH = "claude/epic-fermat-mphgx3"                   # passez à "main" une fois la branche fusionnée
CODE_DIR = "/content/akori_code"
DRIVE_COPY = "/content/drive/MyDrive/AKORI/code/akori.py"   # solution de secours : akori.py copié sur Drive

# Dépôt privé : ajoutez un secret Colab GITHUB_TOKEN (GitHub > Settings > Developer settings > Fine-grained token, lecture seule du dépôt)
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    token = ""

shutil.rmtree(CODE_DIR, ignore_errors=True)
url = f"https://{token + '@' if token else ''}{REPO}"
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE_DIR]).returncode == 0
if not done and os.path.exists(DRIVE_COPY):
    os.makedirs(CODE_DIR, exist_ok=True)
    shutil.copy(DRIVE_COPY, CODE_DIR)
    done = True
    print("Code copié depuis Google Drive.")
if not done:
    from google.colab import files
    os.makedirs(CODE_DIR, exist_ok=True)
    print("Dépôt inaccessible (privé ?). Importez le fichier akori.py (et lui seul).")
    for name, data in files.upload().items():
        if name.endswith(".py"):
            open(os.path.join(CODE_DIR, "akori.py"), "wb").write(data)
assert os.path.exists(os.path.join(CODE_DIR, "akori.py")), "akori.py introuvable : relancez la cellule et importez akori.py"
print("Code prêt :", os.listdir(CODE_DIR))


## 4. Déposer vos PDF

Copiez vos cours dans **`MyDrive/AKORI/a_indexer/`** (le dossier est créé à la première exécution de la cellule 5).

## 5. Build : indexation + pré-génération

In [ ]:
import os, sys
BUILD_USER = "demo@akori.app"          # compte qui contiendra les cours de la démonstration
BUILD_PASSWORD = "ChangezMoi123"       # 8 caractères minimum ; sert à se connecter pendant la démo
LANGS = ("fr", "en")                   # langues des résumés pré-générés

os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
import akori
akori.build_library(user=BUILD_USER, password=BUILD_PASSWORD, langs=LANGS)

## 6. (Optionnel) Régler le seuil de pertinence

AKORI refuse de répondre quand le meilleur passage du cours est trop éloigné de la question. Cette cellule affiche les
scores réels de **vos** questions ; remplacez les exemples par des questions de votre cours et des questions hors sujet.
Les chiffres obtenus sont ceux à citer dans le rapport. Pour fixer le seuil, ajoutez `AKORI_MIN_SIMILARITY` dans la cellule
d'environnement du notebook Demo.

In [ ]:
akori.calibrate_similarity(
    doc_name="MON_COURS.pdf",                       # nom exact du PDF
    relevant=["Définir la mémoire cache L1", "Qu'est-ce que le pipeline ?"],
    offtopic=["Explique le protocole BGP", "Quelle est la capitale du Japon ?"],
    user=BUILD_USER,
)

Terminé ? Ouvrez le notebook **AKORI_Demo.ipynb**.